# 04 — Evaluation, post-processing, and ablation study

This notebook evaluates the cosine-similarity cell-type assignments from Notebook 03 and runs a small ablation study to test whether biological correction improves Cellpose segmentation.

The goal is not just to run one correction setting. We explicitly test multiple post-processing configurations so the final report can say we tried different refinement strategies and measured whether they improved biological consistency.

## What this notebook does

1. Loads the Cellpose baseline outputs.
2. Evaluates marker-gene consistency.
3. Computes confidence and margin scores.
4. Runs several biological correction experiments:
   - baseline / no correction
   - strict correction
   - relaxed correction
   - larger crop correction
   - worst-cells-only correction
5. Compares cosine similarity before vs. after each experiment.
6. Saves metrics, plots, and example corrected cells.


## Imports

In [ ]:
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tifffile as tiff

from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

try:
    from cellpose import models
    CELLPOSE_AVAILABLE = True
except Exception as e:
    CELLPOSE_AVAILABLE = False
    print("Cellpose import failed. Re-segmentation experiments will be skipped.")
    print(e)


## Config

In [ ]:
# =========================
# Config
# =========================

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Core files from Notebooks 02 and 03
CELL_EXPR_PATH = OUTPUT_DIR / "cell_expression_aligned.csv"
PROTOTYPES_PATH = OUTPUT_DIR / "prototypes_normalized.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "cell_type_predictions.csv"
SIM_MATRIX_PATH = OUTPUT_DIR / "cosine_similarity_matrix.csv"
ASSIGNED_TRANSCRIPTS_PATH = OUTPUT_DIR / "assigned_transcripts_crop.csv"

# Segmentation/image files
MASK_PATH = DATA_DIR / "masks_center.npy"
MORPHOLOGY_DIR = DATA_DIR / "morphology_focus"

# Main final outputs
BEST_REFINED_PATH = OUTPUT_DIR / "cell_type_predictions_refined_best.csv"
ALL_RESULTS_PATH = OUTPUT_DIR / "ablation_all_cell_results.csv"
EXPERIMENT_METRICS_PATH = OUTPUT_DIR / "ablation_experiment_metrics.csv"

CROP_SIZE = 512
EXPECTED_SHARED_GENES = 5054
DEFAULT_CONFIDENCE_THRESHOLD = 0.5
DEFAULT_MARGIN_THRESHOLD = 0.05

MARKER_GENES = {
    "fibroblast": ["COL1A1", "COL1A2", "DCN", "LUM", "PDGFRA"],
    "myofibroblast": ["ACTA2", "TAGLN", "MYL9", "COL3A1"],
    "endothelial": ["PECAM1", "VWF", "KDR", "RAMP2", "ENG"],
    "pericyte": ["RGS5", "PDGFRB", "MCAM", "CSPG4"],
    "vascular_smooth_muscle": ["ACTA2", "MYH11", "TAGLN", "MYLK"],
    "cycling": ["MKI67", "TOP2A", "UBE2C", "CENPF"],
    "immune": ["PTPRC", "CD3D", "CD3E", "MS4A1", "LYZ"],
    "epithelial": ["EPCAM", "KRT8", "KRT18", "KRT19"],
}


## Helper functions

In [ ]:
# =========================
# Helper functions
# =========================

def require_file(path: Path, description: str):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}\n"
            "Check the previous notebooks and path configuration."
        )


def savefig(name: str, dpi: int = 300):
    path = OUTPUT_DIR / name
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"Saved → {path}")


def find_first_morphology_image(morphology_dir: Path) -> Path:
    image_files = sorted(
        list(morphology_dir.glob("*.ome.tif")) +
        list(morphology_dir.glob("*.ome.tiff")) +
        list(morphology_dir.glob("*.tif")) +
        list(morphology_dir.glob("*.tiff"))
    )
    if len(image_files) == 0:
        raise FileNotFoundError(f"No TIFF images found in {morphology_dir}")
    return image_files[0]


def parse_cell_id(cell_id_like):
    try:
        return int(str(cell_id_like).replace("cell_", ""))
    except Exception:
        return None


def load_center_crop(image_path: Path, crop_size: int):
    with tiff.TiffFile(image_path) as tif:
        img = tif.pages[0].asarray()
    if img.ndim == 3:
        img = img[..., 0]
    h, w = img.shape
    start_y = h // 2 - crop_size // 2
    start_x = w // 2 - crop_size // 2
    crop = img[start_y:start_y + crop_size, start_x:start_x + crop_size].astype(float)
    crop_norm = (crop - crop.min()) / (crop.max() - crop.min() + 1e-8)
    return crop_norm, start_x, start_y


def compute_margin(similarity_matrix: pd.DataFrame) -> np.ndarray:
    if similarity_matrix.shape[1] < 2:
        raise ValueError("Need at least two prototype columns to compute margin.")
    sorted_scores = np.sort(similarity_matrix.values, axis=1)
    return sorted_scores[:, -1] - sorted_scores[:, -2]


def add_confidence_flags(pred_df, sim_matrix, confidence_threshold, margin_threshold):
    pred_df = pred_df.copy()
    pred_df["confidence"] = pred_df["best_similarity"]
    pred_df["margin"] = compute_margin(sim_matrix)
    pred_df["low_similarity"] = pred_df["confidence"] < confidence_threshold
    pred_df["low_margin"] = pred_df["margin"] < margin_threshold
    pred_df["low_confidence"] = pred_df["low_similarity"] | pred_df["low_margin"]
    return pred_df


@dataclass
class AblationConfig:
    name: str
    confidence_threshold: float
    margin_threshold: float
    patch_size: int
    retry_diameter: int
    accept_tolerance: float
    worst_quantile: Optional[float] = None
    run_resegmentation: bool = True


## 1 · Load data

In [ ]:
# =========================
# 1. Load data
# =========================

require_file(CELL_EXPR_PATH, "cell expression matrix")
require_file(PROTOTYPES_PATH, "normalized prototypes")
require_file(PREDICTIONS_PATH, "cell type predictions")
require_file(SIM_MATRIX_PATH, "cosine similarity matrix")
require_file(MASK_PATH, "Cellpose mask")
require_file(ASSIGNED_TRANSCRIPTS_PATH, "assigned transcript table")

cells = pd.read_csv(CELL_EXPR_PATH, index_col=0)
prototypes = pd.read_csv(PROTOTYPES_PATH, index_col=0)
predictions_raw = pd.read_csv(PREDICTIONS_PATH, index_col=0)
sim_df = pd.read_csv(SIM_MATRIX_PATH, index_col=0)
masks = np.load(MASK_PATH)
assigned_tx = pd.read_csv(ASSIGNED_TRANSCRIPTS_PATH)

print("Cells:", cells.shape)
print("Prototypes:", prototypes.shape)
print("Predictions:", predictions_raw.shape)
print("Similarity matrix:", sim_df.shape)
print("Masks:", masks.shape)
print("Assigned transcripts:", assigned_tx.shape)

if cells.shape[1] != prototypes.shape[1]:
    raise ValueError(f"Gene mismatch: cells have {cells.shape[1]} genes, prototypes have {prototypes.shape[1]} genes.")

if cells.shape[1] != EXPECTED_SHARED_GENES:
    print(f"Warning: expected {EXPECTED_SHARED_GENES} genes, but cell expression has {cells.shape[1]} genes.")

required_tx_cols = {"x_local", "y_local", "feature_name"}
missing_tx_cols = required_tx_cols - set(assigned_tx.columns)
if missing_tx_cols:
    raise ValueError(f"assigned_transcripts_crop.csv is missing columns: {missing_tx_cols}")

common_cells = cells.index.intersection(predictions_raw.index).intersection(sim_df.index)
if len(common_cells) == 0:
    raise ValueError("No overlapping cell IDs among cells, predictions, and similarity matrix. Check cell_1 vs 1 formatting.")

cells = cells.loc[common_cells].copy()
predictions_raw = predictions_raw.loc[common_cells].copy()
sim_df = sim_df.loc[common_cells].copy()

print("\nAfter cell-index alignment:")
print("Cells:", cells.shape)
print("Predictions:", predictions_raw.shape)
print("Similarity matrix:", sim_df.shape)

print("\nPredicted cell type counts:")
print(predictions_raw["predicted_cell_type"].value_counts())


## 2 · Load image crop for spatial plots and correction

In [ ]:
# =========================
# 2. Load image crop
# =========================

IMAGE_PATH = find_first_morphology_image(MORPHOLOGY_DIR)
print(f"Using image: {IMAGE_PATH}")
img_crop, start_x, start_y = load_center_crop(IMAGE_PATH, CROP_SIZE)
print("Image crop shape:", img_crop.shape)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img_crop, cmap="gray")
ax.set_title("Center crop used for Cellpose and transcript assignment")
ax.axis("off")
plt.tight_layout()
savefig("evaluation_center_crop.png")
plt.show()


## 3 · Marker gene evaluation

In [ ]:
# =========================
# 3. Marker gene evaluation
# =========================

available_markers = {}
for marker_group, genes in MARKER_GENES.items():
    present = [g for g in genes if g in cells.columns]
    missing = [g for g in genes if g not in cells.columns]
    if present:
        available_markers[marker_group] = present
    if missing:
        print(f"{marker_group}: markers not in panel/expression matrix: {missing}")

print("\nMarkers available for evaluation:")
for marker_group, genes in available_markers.items():
    print(f"{marker_group}: {genes}")

if len(available_markers) == 0:
    raise ValueError("No marker genes were found in the cell expression matrix.")

cells_with_pred = cells.copy()
cells_with_pred["predicted_cell_type"] = predictions_raw["predicted_cell_type"]
rows = []
for marker_group, genes in available_markers.items():
    mean_expr = cells_with_pred.groupby("predicted_cell_type")[genes].mean().mean(axis=1)
    mean_expr.name = marker_group
    rows.append(mean_expr)

marker_df = pd.DataFrame(rows).fillna(0)
marker_df_norm = marker_df.div(marker_df.max(axis=1) + 1e-9, axis=0)
print("Marker matrix:", marker_df.shape)
display(marker_df)
display(marker_df_norm)


In [ ]:
# =========================
# 3b. Marker heatmap
# =========================

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(marker_df_norm, cmap="YlOrRd", ax=ax, linewidths=0.4, annot=marker_df.round(2), fmt=".2f", annot_kws={"size": 7})
ax.set_title("Marker gene expression per predicted cell type\n(relative row-normalized enrichment)")
ax.set_xlabel("Predicted cell type")
ax.set_ylabel("Marker gene group")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
savefig("marker_gene_heatmap.png")
plt.show()


## 4 · Baseline confidence scoring

In [ ]:
# =========================
# 4. Baseline confidence scoring
# =========================

predictions_baseline_abs = add_confidence_flags(predictions_raw, sim_df, DEFAULT_CONFIDENCE_THRESHOLD, DEFAULT_MARGIN_THRESHOLD)
percentile_threshold = predictions_raw["best_similarity"].quantile(0.25)
predictions = add_confidence_flags(predictions_raw, sim_df, percentile_threshold, DEFAULT_MARGIN_THRESHOLD)

print("Absolute-threshold QC:")
print(f"  confidence threshold = {DEFAULT_CONFIDENCE_THRESHOLD:.4f}")
print(f"  low-confidence cells = {predictions_baseline_abs['low_confidence'].sum()} / {len(predictions_baseline_abs)}")
print("\nPercentile-threshold QC used for ablation:")
print(f"  25th percentile confidence threshold = {percentile_threshold:.4f}")
print(f"  margin threshold = {DEFAULT_MARGIN_THRESHOLD:.4f}")
print(f"  low-confidence cells = {predictions['low_confidence'].sum()} / {len(predictions)}")
print("\nLow-confidence breakdown by predicted type:")
print(predictions[predictions["low_confidence"]]["predicted_cell_type"].value_counts())
display(predictions.head())


In [ ]:
# =========================
# 4b. Confidence and margin histograms
# =========================

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(predictions["confidence"], bins=30, edgecolor="white", linewidth=0.4)
ax.axvline(percentile_threshold, color="red", linestyle="--", label=f"25th percentile = {percentile_threshold:.3f}")
ax.axvline(DEFAULT_CONFIDENCE_THRESHOLD, color="black", linestyle=":", label=f"absolute = {DEFAULT_CONFIDENCE_THRESHOLD}")
ax.set_xlabel("Best cosine similarity")
ax.set_ylabel("Number of cells")
ax.set_title("Assignment confidence")
ax.legend()
plt.tight_layout()
savefig("confidence_histogram.png")
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(predictions["margin"], bins=30, edgecolor="white", linewidth=0.4)
ax.axvline(DEFAULT_MARGIN_THRESHOLD, color="red", linestyle="--", label=f"threshold = {DEFAULT_MARGIN_THRESHOLD}")
ax.set_xlabel("Score margin (best − second best)")
ax.set_ylabel("Number of cells")
ax.set_title("Assignment ambiguity")
ax.legend()
plt.tight_layout()
savefig("margin_histogram.png")
plt.show()


## 5 · Biological correction function

In [ ]:
# =========================
# 5. Biological correction function
# =========================

def _rejection_row(config, cell_id_str, before_score, before_type, decision, extra=None):
    row = {
        "experiment": config.name,
        "cell_id": cell_id_str,
        "before_score": before_score,
        "candidate_after_score": before_score,
        "after_score": before_score,
        "before_type": before_type,
        "candidate_after_type": before_type,
        "after_type": before_type,
        "accepted": False,
        "decision": decision,
        "improvement": 0.0,
        "raw_candidate_improvement": 0.0,
        "patch_size": config.patch_size,
        "retry_diameter": config.retry_diameter,
    }
    if extra:
        row.update(extra)
    return row


def run_biological_correction_experiment(config, predictions_input, sim_matrix, masks, img_crop, assigned_tx, prototypes):
    """
    Re-run Cellpose locally around selected low-confidence cells and accept a candidate
    correction only if cosine similarity improves by at least config.accept_tolerance.
    """
    predictions_flagged = add_confidence_flags(
        pred_df=predictions_input,
        sim_matrix=sim_matrix,
        confidence_threshold=config.confidence_threshold,
        margin_threshold=config.margin_threshold,
    )

    if config.worst_quantile is not None:
        cutoff = predictions_flagged["best_similarity"].quantile(config.worst_quantile)
        selected = predictions_flagged[predictions_flagged["best_similarity"] <= cutoff].copy()
    else:
        selected = predictions_flagged[predictions_flagged["low_confidence"]].copy()

    print(f"\nRunning experiment: {config.name}")
    print(f"Selected cells: {len(selected)} / {len(predictions_flagged)}")
    print(f"Patch size: {config.patch_size}")
    print(f"Retry diameter: {config.retry_diameter}")
    print(f"Accept tolerance: {config.accept_tolerance}")

    refined_predictions = predictions_flagged.copy()
    ablation_rows = []
    proto_values = prototypes.values
    shared_genes = list(prototypes.columns)

    if not config.run_resegmentation or not CELLPOSE_AVAILABLE:
        print("Re-segmentation unavailable/skipped. Creating no-correction table.")
        for cell_id_str, row in selected.iterrows():
            ablation_rows.append(_rejection_row(config, cell_id_str, row["best_similarity"], row["predicted_cell_type"], "not_run"))
        return refined_predictions, pd.DataFrame(ablation_rows)

    model = models.CellposeModel(gpu=True)

    for cell_id_str, row in selected.iterrows():
        cell_id = parse_cell_id(cell_id_str)
        if cell_id is None:
            continue

        before_score = row["best_similarity"]
        before_type = row["predicted_cell_type"]
        ys, xs = np.where(masks == cell_id)

        if len(ys) == 0:
            ablation_rows.append(_rejection_row(config, cell_id_str, before_score, before_type, "rejected_no_mask_pixels"))
            continue

        cy, cx = int(ys.mean()), int(xs.mean())
        y0 = max(0, cy - config.patch_size // 2)
        y1 = min(CROP_SIZE, cy + config.patch_size // 2)
        x0 = max(0, cx - config.patch_size // 2)
        x1 = min(CROP_SIZE, cx + config.patch_size // 2)
        patch = img_crop[y0:y1, x0:x1]

        if patch.size == 0:
            ablation_rows.append(_rejection_row(config, cell_id_str, before_score, before_type, "rejected_empty_patch"))
            continue

        patch_masks, _, _ = model.eval(patch, diameter=config.retry_diameter, batch_size=1)
        patch_cy = (y1 - y0) // 2
        patch_cx = (x1 - x0) // 2
        new_cell_id = patch_masks[patch_cy, patch_cx]

        if new_cell_id == 0:
            ablation_rows.append(_rejection_row(config, cell_id_str, before_score, before_type, "rejected_no_center_cell", {"x0":x0,"x1":x1,"y0":y0,"y1":y1}))
            continue

        patch_tx = assigned_tx[
            (assigned_tx["x_local"] >= x0) & (assigned_tx["x_local"] < x1) &
            (assigned_tx["y_local"] >= y0) & (assigned_tx["y_local"] < y1)
        ].copy()

        if len(patch_tx) == 0:
            ablation_rows.append(_rejection_row(config, cell_id_str, before_score, before_type, "rejected_no_transcripts", {"x0":x0,"x1":x1,"y0":y0,"y1":y1}))
            continue

        patch_tx["patch_cell"] = patch_masks[
            (patch_tx["y_local"] - y0).astype(int).clip(0, patch_masks.shape[0] - 1),
            (patch_tx["x_local"] - x0).astype(int).clip(0, patch_masks.shape[1] - 1),
        ]
        new_cell_tx = patch_tx[patch_tx["patch_cell"] == new_cell_id]

        if len(new_cell_tx) == 0:
            ablation_rows.append(_rejection_row(config, cell_id_str, before_score, before_type, "rejected_no_transcripts_in_new_cell", {"x0":x0,"x1":x1,"y0":y0,"y1":y1}))
            continue

        new_expr = new_cell_tx["feature_name"].value_counts().reindex(shared_genes, fill_value=0).values.reshape(1, -1)
        if new_expr.sum() == 0:
            ablation_rows.append(_rejection_row(config, cell_id_str, before_score, before_type, "rejected_zero_expression", {"x0":x0,"x1":x1,"y0":y0,"y1":y1}))
            continue

        new_expr_norm = normalize(new_expr, norm="l2")
        scores = cos_sim(new_expr_norm, proto_values)[0]
        candidate_after_score = float(scores.max())
        candidate_after_type = prototypes.index[scores.argmax()]
        candidate_margin = float(np.sort(scores)[-1] - np.sort(scores)[-2])
        raw_improvement = candidate_after_score - before_score
        accepted = candidate_after_score >= before_score + config.accept_tolerance

        if accepted:
            refined_predictions.loc[cell_id_str, "predicted_cell_type"] = candidate_after_type
            refined_predictions.loc[cell_id_str, "best_similarity"] = candidate_after_score
            refined_predictions.loc[cell_id_str, "confidence"] = candidate_after_score
            refined_predictions.loc[cell_id_str, "margin"] = candidate_margin
            refined_predictions.loc[cell_id_str, "low_similarity"] = candidate_after_score < config.confidence_threshold
            refined_predictions.loc[cell_id_str, "low_margin"] = candidate_margin < config.margin_threshold
            refined_predictions.loc[cell_id_str, "low_confidence"] = (
                refined_predictions.loc[cell_id_str, "low_similarity"] or
                refined_predictions.loc[cell_id_str, "low_margin"]
            )

        ablation_rows.append({
            "experiment": config.name,
            "cell_id": cell_id_str,
            "before_score": before_score,
            "candidate_after_score": candidate_after_score,
            "after_score": candidate_after_score if accepted else before_score,
            "before_type": before_type,
            "candidate_after_type": candidate_after_type,
            "after_type": candidate_after_type if accepted else before_type,
            "accepted": accepted,
            "decision": "accepted_improved" if accepted else "rejected_no_improvement",
            "improvement": raw_improvement if accepted else 0.0,
            "raw_candidate_improvement": raw_improvement,
            "patch_size": config.patch_size,
            "retry_diameter": config.retry_diameter,
            "x0": x0,
            "x1": x1,
            "y0": y0,
            "y1": y1,
        })

    return refined_predictions, pd.DataFrame(ablation_rows)


## 6 · Define ablation experiments

In [ ]:
# =========================
# 6. Define ablation experiments
# =========================

experiment_configs = [
    AblationConfig("baseline_no_correction", percentile_threshold, DEFAULT_MARGIN_THRESHOLD, 80, 20, 0.0, run_resegmentation=False),
    AblationConfig("strict_patch80_diam20", percentile_threshold, DEFAULT_MARGIN_THRESHOLD, 80, 20, 1e-4),
    AblationConfig("relaxed_patch80_diam20", percentile_threshold, DEFAULT_MARGIN_THRESHOLD, 80, 20, 0.0),
    AblationConfig("larger_patch150_diam20", percentile_threshold, DEFAULT_MARGIN_THRESHOLD, 150, 20, 0.0),
    AblationConfig("worst25_patch150_diam20", percentile_threshold, DEFAULT_MARGIN_THRESHOLD, 150, 20, 0.0, worst_quantile=0.25),
]

pd.DataFrame([cfg.__dict__ for cfg in experiment_configs])


## 7 · Run ablation experiments

In [ ]:
# =========================
# 7. Run ablation experiments
# =========================

all_ablation_results = []
refined_by_experiment = {}

for cfg in experiment_configs:
    refined_pred, ablation_df = run_biological_correction_experiment(
        config=cfg,
        predictions_input=predictions_raw,
        sim_matrix=sim_df,
        masks=masks,
        img_crop=img_crop,
        assigned_tx=assigned_tx,
        prototypes=prototypes,
    )
    refined_by_experiment[cfg.name] = refined_pred
    if len(ablation_df) > 0:
        all_ablation_results.append(ablation_df)

if len(all_ablation_results) == 0:
    raise ValueError("No ablation results were produced.")

all_results_df = pd.concat(all_ablation_results, ignore_index=True)
all_results_df.to_csv(ALL_RESULTS_PATH, index=False)

print(f"Saved all ablation cell-level results → {ALL_RESULTS_PATH}")
print("\nDecision counts by experiment:")
display(pd.crosstab(all_results_df["experiment"], all_results_df["decision"]))


## 8 · Summarize ablation metrics

In [ ]:
# =========================
# 8. Summarize ablation metrics
# =========================

def summarize_experiment(name, refined_pred, ablation_df):
    before_scores = predictions_raw["best_similarity"]
    after_scores = refined_pred["best_similarity"]
    before_flagged = add_confidence_flags(predictions_raw, sim_df, percentile_threshold, DEFAULT_MARGIN_THRESHOLD)

    if "margin" in refined_pred.columns:
        after_low = (refined_pred["best_similarity"] < percentile_threshold) | (refined_pred["margin"] < DEFAULT_MARGIN_THRESHOLD)
    else:
        after_low = refined_pred["best_similarity"] < percentile_threshold

    diff = after_scores - before_scores
    n_total = len(predictions_raw)
    n_considered = len(ablation_df)
    n_accepted = int(ablation_df["accepted"].sum()) if len(ablation_df) else 0
    n_rejected = n_considered - n_accepted
    improved_scores = ablation_df.loc[ablation_df["accepted"], "improvement"] if len(ablation_df) else pd.Series(dtype=float)

    return {
        "experiment": name,
        "total_segmented_cells": n_total,
        "mean_cosine_before": before_scores.mean(),
        "mean_cosine_after": after_scores.mean(),
        "median_cosine_before": before_scores.median(),
        "median_cosine_after": after_scores.median(),
        "std_cosine_before": before_scores.std(),
        "std_cosine_after": after_scores.std(),
        "low_confidence_before": int(before_flagged["low_confidence"].sum()),
        "low_confidence_after": int(after_low.sum()),
        "percent_low_confidence_before": 100 * before_flagged["low_confidence"].sum() / n_total,
        "percent_low_confidence_after": 100 * after_low.sum() / n_total,
        "cells_considered_for_correction": n_considered,
        "cells_improved": n_accepted,
        "percent_improved_of_considered": 100 * n_accepted / max(n_considered, 1),
        "cells_rejected_or_unchanged": n_rejected,
        "average_cosine_improvement_among_improved": improved_scores.mean() if len(improved_scores) else 0,
        "max_cosine_improvement": improved_scores.max() if len(improved_scores) else 0,
        "overall_mean_delta": diff.mean(),
        "overall_max_delta": diff.max(),
        "cells_with_any_score_change": int((diff != 0).sum()),
    }

metrics_rows = []
for cfg in experiment_configs:
    refined_pred = refined_by_experiment[cfg.name]
    ablation_df = all_results_df[all_results_df["experiment"] == cfg.name].copy()
    metrics_rows.append(summarize_experiment(cfg.name, refined_pred, ablation_df))

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(EXPERIMENT_METRICS_PATH, index=False)
print(f"Saved experiment metrics → {EXPERIMENT_METRICS_PATH}")
display(metrics_df)


In [ ]:
# =========================
# 8b. Pick best experiment
# =========================

best_row = metrics_df.sort_values(["overall_mean_delta", "cells_improved", "max_cosine_improvement"], ascending=False).iloc[0]
BEST_EXPERIMENT = best_row["experiment"]
best_refined_predictions = refined_by_experiment[BEST_EXPERIMENT].copy()
best_ablation_df = all_results_df[all_results_df["experiment"] == BEST_EXPERIMENT].copy()
best_refined_predictions.to_csv(BEST_REFINED_PATH)

print("Best experiment:", BEST_EXPERIMENT)
display(best_row.to_frame(name="value"))
print(f"Saved best refined predictions → {BEST_REFINED_PATH}")


## 9 · Visualize ablation experiment results

In [ ]:
# =========================
# 9. Experiment summary plot
# =========================

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=metrics_df, x="experiment", y="cells_improved", ax=axes[0])
axes[0].set_title("Cells improved")
axes[0].set_xlabel("")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=35)

sns.barplot(data=metrics_df, x="experiment", y="overall_mean_delta", ax=axes[1])
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Overall mean cosine change")
axes[1].set_xlabel("")
axes[1].set_ylabel("After - before")
axes[1].tick_params(axis="x", rotation=35)

sns.barplot(data=metrics_df, x="experiment", y="max_cosine_improvement", ax=axes[2])
axes[2].set_title("Maximum accepted improvement")
axes[2].set_xlabel("")
axes[2].set_ylabel("Max improvement")
axes[2].tick_params(axis="x", rotation=35)
plt.tight_layout()
savefig("ablation_experiment_summary.png")
plt.show()


## 10 · Before/after plots for best experiment

In [ ]:
# =========================
# 10. Before / after comparison for best experiment
# =========================

comparison_df = pd.DataFrame({
    "before": predictions_raw["best_similarity"],
    "after": best_refined_predictions["best_similarity"],
})
comparison_long = comparison_df.reset_index().melt(id_vars="index", value_vars=["before", "after"], var_name="condition", value_name="cosine_similarity")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(comparison_df["before"], bins=30, alpha=0.6, label="Before")
ax.hist(comparison_df["after"], bins=30, alpha=0.6, label="After")
ax.axvline(percentile_threshold, color="red", linestyle="--", label="25th percentile threshold")
ax.set_xlabel("Best cosine similarity")
ax.set_ylabel("Number of cells")
ax.set_title(f"Cosine similarity before vs after\nBest experiment: {BEST_EXPERIMENT}")
ax.legend()
plt.tight_layout()
savefig("cosine_before_after_best_histogram.png")
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=comparison_long, x="condition", y="cosine_similarity", ax=ax)
ax.set_title(f"Cosine similarity before vs after\nBest experiment: {BEST_EXPERIMENT}")
ax.set_xlabel("")
ax.set_ylabel("Best cosine similarity")
plt.tight_layout()
savefig("cosine_before_after_best_boxplot.png")
plt.show()


In [ ]:
# =========================
# 10b. Improvement distribution
# =========================

diff = best_refined_predictions["best_similarity"] - predictions_raw["best_similarity"]
print(diff.describe())
print("Cells improved:", (diff > 0).sum())
print("Cells unchanged:", (diff == 0).sum())
print("Mean improvement:", diff.mean())
print("Max improvement:", diff.max())

plt.figure(figsize=(6, 4))
plt.hist(diff, bins=30)
plt.axvline(0, color="red", linestyle="--")
plt.title(f"Cosine similarity improvement\nBest experiment: {BEST_EXPERIMENT}")
plt.xlabel("After - before cosine similarity")
plt.ylabel("Number of cells")
plt.tight_layout()
savefig("cosine_improvement_distribution_best.png")
plt.show()


## 11 · Spatial confidence map

In [ ]:
# =========================
# 11. Spatial confidence map
# =========================

best_predictions_with_flags = add_confidence_flags(best_refined_predictions, sim_df, percentile_threshold, DEFAULT_MARGIN_THRESHOLD)
confidence_map = np.zeros(masks.shape, dtype=float)
margin_map = np.zeros(masks.shape, dtype=float)
low_conf_map = np.zeros(masks.shape, dtype=float)

for cell_id_str, row in best_predictions_with_flags.iterrows():
    cell_id = parse_cell_id(cell_id_str)
    if cell_id is None:
        continue
    confidence_map[masks == cell_id] = row["confidence"]
    margin_map[masks == cell_id] = row["margin"]
    low_conf_map[masks == cell_id] = 1.0 if row["low_confidence"] else 0.0

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_crop, cmap="gray")
levels = np.unique(masks)
levels = levels[levels > 0]
if len(levels):
    axes[0].contour(masks, levels=levels, linewidths=0.3, colors="cyan", alpha=0.5)
axes[0].set_title("Cellpose segmentation")
axes[0].axis("off")

axes[1].imshow(img_crop, cmap="gray")
conf_masked = np.ma.masked_where(masks == 0, confidence_map)
im = axes[1].imshow(conf_masked, cmap="RdYlGn", vmin=0, vmax=max(0.2, confidence_map.max()), alpha=0.75)
plt.colorbar(im, ax=axes[1], fraction=0.046, label="Best cosine similarity")
axes[1].set_title("Per-cell confidence after best correction")
axes[1].axis("off")
plt.tight_layout()
savefig("confidence_map.png")
plt.show()


## 12 · Flagged cells overlay

In [ ]:
# =========================
# 12. Flagged cells overlay
# =========================

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(img_crop, cmap="gray")
for cell_id_str, row in best_predictions_with_flags.iterrows():
    cell_id = parse_cell_id(cell_id_str)
    if cell_id is None:
        continue
    ys, xs = np.where(masks == cell_id)
    if len(ys) == 0:
        continue
    color = "red" if row["low_confidence"] else "lime"
    ax.plot(xs.mean(), ys.mean(), ".", color=color, markersize=5, alpha=0.85)

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0], [0], marker=".", color="w", markerfacecolor="lime", markersize=8, label="higher confidence"),
    Line2D([0], [0], marker=".", color="w", markerfacecolor="red", markersize=8, label="low confidence"),
], loc="upper right", fontsize=8)
ax.set_title("Flagged low-confidence cells after best correction")
ax.axis("off")
plt.tight_layout()
savefig("flagged_cells_overlay.png")
plt.show()


## 13 · Example corrected cells

In [ ]:
# =========================
# 13. Example corrected cells
# =========================

accepted_examples = best_ablation_df[best_ablation_df["accepted"]].copy()
accepted_examples = accepted_examples.sort_values("improvement", ascending=False).head(6)

if len(accepted_examples) == 0:
    print("No accepted examples found for the best experiment.")
else:
    n = len(accepted_examples)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    if n == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, accepted_examples.iterrows()):
        cell_id_str = row["cell_id"]
        cell_id = parse_cell_id(cell_id_str)
        ys, xs = np.where(masks == cell_id)
        if len(ys) == 0:
            ax.axis("off")
            continue
        cy, cx = int(ys.mean()), int(xs.mean())
        patch_size = int(row.get("patch_size", 120))
        y0 = max(0, cy - patch_size // 2)
        y1 = min(CROP_SIZE, cy + patch_size // 2)
        x0 = max(0, cx - patch_size // 2)
        x1 = min(CROP_SIZE, cx + patch_size // 2)
        patch = img_crop[y0:y1, x0:x1]
        patch_mask = masks[y0:y1, x0:x1] == cell_id
        ax.imshow(patch, cmap="gray")
        ax.contour(patch_mask, levels=[0.5], colors="cyan", linewidths=0.8)
        title = f"{cell_id_str}\n{row['before_type']} → {row['after_type']}\n{row['before_score']:.3f} → {row['after_score']:.3f}"
        ax.set_title(title, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    savefig("example_corrected_cells.png")
    plt.show()

display(accepted_examples)


## 14 · Final sanity checks and report summary

In [ ]:
# =========================
# 14. Final sanity checks
# =========================

print("Notebook 04 sanity check")
print("------------------------")
print(f"Cells evaluated:                  {len(cells):,}")
print(f"Genes per cell:                   {cells.shape[1]:,}")
print(f"Prototype cell types:             {prototypes.shape[0]:,}")
print(f"Prediction rows:                  {len(predictions_raw):,}")
print(f"Best experiment:                  {BEST_EXPERIMENT}")
print(f"Best refined prediction path:     {BEST_REFINED_PATH}")
print(f"All cell-level results path:      {ALL_RESULTS_PATH}")
print(f"Experiment metrics path:          {EXPERIMENT_METRICS_PATH}")
print(f"Marker heatmap path:              {OUTPUT_DIR / 'marker_gene_heatmap.png'}")
print(f"Confidence map path:              {OUTPUT_DIR / 'confidence_map.png'}")
print(f"Flagged overlay path:             {OUTPUT_DIR / 'flagged_cells_overlay.png'}")

assert len(cells) == len(predictions_raw)
assert cells.shape[1] == prototypes.shape[1]
assert BEST_REFINED_PATH.exists()
assert ALL_RESULTS_PATH.exists()
assert EXPERIMENT_METRICS_PATH.exists()
print("\nNotebook 04 completed successfully.")


In [ ]:
# =========================
# 14b. Short result summary for report draft
# =========================

best_metrics = metrics_df[metrics_df["experiment"] == BEST_EXPERIMENT].iloc[0]
print("Report-ready summary:")
print("---------------------")
print(f"We evaluated {int(best_metrics['total_segmented_cells'])} segmented cells using {cells.shape[1]} shared genes and {prototypes.shape[0]} scRNA-seq-derived cell-type prototypes.")
print(f"The best correction setting was '{BEST_EXPERIMENT}', which improved {int(best_metrics['cells_improved'])} out of {int(best_metrics['cells_considered_for_correction'])} considered low-confidence cells.")
print(f"Mean cosine similarity changed from {best_metrics['mean_cosine_before']:.4f} to {best_metrics['mean_cosine_after']:.4f}.")
print(f"The average accepted-cell improvement was {best_metrics['average_cosine_improvement_among_improved']:.4f}, with a maximum improvement of {best_metrics['max_cosine_improvement']:.4f}.")
print("This suggests that the biological correction pipeline can improve a subset of low-confidence cells, but the global shift in cosine similarity may remain modest.")
